# AllSortsHub Cartoon Studio — Wan 2.2 Colab Generator

This notebook runs **Wan 2.2 Image-to-Video locally on a Colab GPU**, instead of using the Hugging Face ZeroGPU Space or Magic Hour credits. It uses the existing Episode 1 storyboard, prompts, and audio in this repository.

> **GPU:** Wan 2.2 I2V A14B is a large model. Use a Colab **L4/A100 (24GB+ recommended; A100 preferred)**. A free T4 may not have enough VRAM. The official Wan documentation also lists 24GB+ with offloading as the practical lower range.

The notebook is checkpointed: completed `shot_XX.mp4` / `shot_XXa.mp4` / `shot_XXb.mp4` files are skipped on later runs. Generated media is stored in Google Drive so a disconnected Colab session can be resumed.

In [ ]:
# 1. Check GPU
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

In [ ]:
# 2. Mount Google Drive for persistent checkpoints/output
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/AllSortsHub-Wan2.2'
!mkdir -p "$BASE"

In [ ]:
# 3. Clone/update the cartoon project and official Wan 2.2 implementation
%cd /content
!git clone https://github.com/parth01/AllSortsHub-Cartoon-Studio.git cartoon-studio 2>/dev/null || (cd cartoon-studio && git pull)
!git clone https://github.com/Wan-Video/Wan2.2.git Wan2.2 2>/dev/null || (cd Wan2.2 && git pull)
%cd /content/Wan2.2
!pip install -q -r requirements.txt
!pip install -q modelscope huggingface_hub transformers accelerate

## 4. Download the model

This uses the ModelScope copy of Wan 2.2 so generation is not dependent on your Hugging Face ZeroGPU allowance. The model is large, so keep it on Google Drive between sessions.

In [ ]:
MODEL_DIR = '/content/drive/MyDrive/AllSortsHub-Wan2.2/Wan2.2-I2V-A14B'
import os
os.makedirs(MODEL_DIR, exist_ok=True)
from modelscope import snapshot_download
snapshot_download('Wan-AI/Wan2.2-I2V-A14B', local_dir=MODEL_DIR)
print('Model ready:', MODEL_DIR)

In [ ]:
# 5. Copy the Episode 1 project into the working tree
%cd /content/cartoon-studio/master-version/AllSortsHub-Billion 2
!mkdir -p wan_i2v/generated output
print('Project ready')
!ls -lh wan_i2v/manifest.json wan_i2v/prompts.txt

In [ ]:
# 6. Run the resumable Episode 1 generator
# It generates one shot at a time. Re-run this cell after a disconnect; existing clips are skipped.
import json, os, subprocess, sys
from pathlib import Path
ROOT = Path('/content/cartoon-studio/master-version/AllSortsHub-Billion 2')
WAN = Path('/content/Wan2.2')
GEN = ROOT / 'wan_i2v' / 'generated'
with open(ROOT / 'wan_i2v' / 'manifest.json') as f: manifest = json.load(f)
prompts = (ROOT / 'wan_i2v' / 'prompts.txt').read_text()
STYLE = 'Modern 2D cel-shaded cartoon animation, bold clean black linework, semi-flat shading, vibrant colors, expressive anime-influenced facial acting, preserve the exact character designs and environment in the input image. Natural hand-drawn animation feel. Keep faces, hair, clothing, proportions, props and background layout consistent with the source frame. Motion should be smooth, readable and physically plausible. No new characters, no redesign, no text changes.'
NEG = 'No photorealism, no 3D CGI, no live action, no extra fingers, no duplicate limbs, no warped faces, no character morphing, no costume changes, no hairstyle changes, no background replacement, no random objects, no random text, no logos, no watermark, no scene cuts inside the generated clip, no sudden camera spins, no extreme deformation.'
def prompt_for(n):
    marker = f'SHOT {n:02d} —'
    start = prompts.find(marker)
    if start < 0: raise RuntimeError('Missing prompt for ' + marker)
    end = prompts.find('\n\nSHOT ', start + 2)
    if end < 0: end = prompts.find('\n\nNEGATIVE', start + 2)
    section = prompts[start:end if end >= 0 else None].split('\n', 1)[1].strip()
    return f'{STYLE} {section} {NEG}'
def run_clip(image, prompt, seconds, out, seed):
    if out.exists() and out.stat().st_size > 10000:
        print('SKIP', out.name); return
    frames = 81  # 5 seconds at Wan 2.2's 16 fps-equivalent sample cadence
    cmd = [sys.executable, str(WAN/'generate.py'), '--task', 'i2v-A14B', '--size', '832*480', '--ckpt_dir', MODEL_DIR, '--offload_model', 'True', '--convert_model_dtype', '--t5_cpu', '--frame_num', str(frames), '--image', str(image), '--prompt', prompt, '--base_seed', str(seed), '--save_file', str(out)]
    print('GENERATE', out.name, 'seed', seed)
    subprocess.run(cmd, cwd=WAN, check=True)
for shot in manifest['shots']:
    n = int(shot['id']); duration = float(shot['duration']); image = ROOT / shot['image']
    p = prompt_for(n)
    if duration <= 5:
        run_clip(image, p, duration, GEN/f'shot_{n:02d}.mp4', 910000+n)
    else:
        first = GEN/f'shot_{n:02d}a.mp4'; run_clip(image, p, 5, first, 910000+n)
        frame = GEN/f'shot_{n:02d}_continuation.jpg'
        if not (GEN/f'shot_{n:02d}b.mp4').exists() or (GEN/f'shot_{n:02d}b.mp4').stat().st_size <= 10000:
            subprocess.run(['ffmpeg','-y','-sseof','-0.08','-i',str(first),'-frames:v','1','-q:v','2',str(frame)], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            run_clip(frame, f'{STYLE} Continue the exact scene from the supplied final frame. Continue naturally from the previous motion without resetting the characters, camera, lighting, props, clothing, hair, or background. {p} {NEG}', duration-5, GEN/f'shot_{n:02d}b.mp4', 1010000+n)
            frame.unlink(missing_ok=True)
print('Generation pass complete.')

In [ ]:
# 7. Assemble the complete Episode 1 master + vertical video
%cd /content/cartoon-studio/master-version/AllSortsHub-Billion 2
!python3 wan_i2v/assemble_episode.py
!ls -lh output/AllSortsHub_Episode_01_WAN_MASTER.mp4 output/AllSortsHub_Episode_01_WAN_VERTICAL_9x16.mp4

## 8. Download / save

The finished files are in `output/`. You can download them from the Colab file browser, or copy them to Google Drive.

If Colab disconnects during generation, **do not delete the Drive folder**. Reconnect, run the setup cells as needed, and rerun the generation cell. Existing completed shots will be skipped.